# 🚀 RAG Workshop - Environment Setup

Welcome to the **RAG & Multimodal Knowledge Workshop**! This notebook will guide you through setting up all required Azure resources.

## What This Notebook Does

1. **Checks prerequisites** - Python version, required tools
2. **Guides Azure deployment** - Two options: automated or manual
3. **Generates your `.env` file** - All connection strings in one place
4. **Validates your setup** - Quick connectivity tests

## Prerequisites

Before starting, ensure you have:
- ✅ An Azure subscription with **Owner** or **Contributor** role
- ✅ Python 3.11+ installed
- ✅ Azure CLI installed (`az --version`)

---

**Estimated time**: ~20 minutes

## Step 1: Check Prerequisites

Let's verify your environment is ready for the workshop.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

print("🔍 Checking Prerequisites...")
print("=" * 50)

# Check Python version
python_version = sys.version_info
python_ok = python_version >= (3, 11) and python_version < (3, 14)
print(f"\n📌 Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")
if python_ok:
    print("   ✅ Python version is compatible (≥3.11, <3.14)")
else:
    print("   ❌ Python version must be ≥3.11 and <3.14")
    print("   💡 Install Python 3.11 or 3.12: brew install python@3.11 (macOS)")

# Check Azure CLI
az_available = shutil.which("az") is not None
print(f"\n📌 Azure CLI: {'Found' if az_available else 'Not found'}")
if az_available:
    try:
        result = subprocess.run(["az", "--version"], capture_output=True, text=True)
        az_version_line = result.stdout.split("\n")[0]
        print(f"   ✅ {az_version_line}")
    except Exception:
        print("   ✅ Azure CLI is available")
else:
    print("   ❌ Azure CLI not found")
    print("   💡 Install: https://aka.ms/installazurecli")

# Check jq (needed for deployment script)
jq_available = shutil.which("jq") is not None
print(f"\n📌 jq (JSON processor): {'Found' if jq_available else 'Not found'}")
if jq_available:
    print("   ✅ jq is available")
else:
    print("   ⚠️ jq not found (optional, used by deploy script)")
    print("   💡 Install: brew install jq (macOS) or apt install jq (Linux)")

# Check poppler (for PDF processing)
pdftoppm_available = shutil.which("pdftoppm") is not None
print(f"\n📌 Poppler (PDF tools): {'Found' if pdftoppm_available else 'Not found'}")
if pdftoppm_available:
    print("   ✅ Poppler is available")
else:
    print("   ⚠️ Poppler not found (needed for PDF figure extraction in later modules)")
    print("   💡 Install: brew install poppler (macOS) or apt install poppler-utils (Linux)")

# Summary
print("\n" + "=" * 50)
all_ok = python_ok and az_available
if all_ok:
    print("🎉 All required prerequisites are met!")
    print("\n➡️ Proceed to Step 2: Azure Setup")
else:
    print("⚠️ Some prerequisites are missing. Please install them before continuing.")

## Step 2: Azure Login & Subscription

Let's ensure you're logged into Azure and using the correct subscription.

In [ ]:
import subprocess
import json

print("🔐 Checking Azure Login Status...")
print("=" * 50)

try:
    # Check if logged in
    result = subprocess.run(
        ["az", "account", "show"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        account = json.loads(result.stdout)
        print(f"\n✅ Logged in as: {account.get('user', {}).get('name', 'Unknown')}")
        print(f"📌 Current Subscription: {account.get('name', 'Unknown')}")
        print(f"   ID: {account.get('id', 'Unknown')}")
        print(f"   State: {account.get('state', 'Unknown')}")
        
        # Store for later use
        SUBSCRIPTION_ID = account.get('id')
        print(f"\n➡️ If this is correct, proceed to Step 3")
        print(f"   To change subscription, run: az account set --subscription <subscription-id>")
    else:
        print("\n❌ Not logged in to Azure")
        print("\n💡 Run the cell below to login:")
        
except FileNotFoundError:
    print("\n❌ Azure CLI not found. Please install it first.")

In [ ]:
# Run this cell ONLY if you need to login to Azure
# This will open a browser window for authentication

!az login

In [ ]:
# Run this cell to list all available subscriptions
# Then use the next cell to set the correct one

!az account list --output table

In [ ]:
# Uncomment and run this cell to change your active subscription
# Replace <subscription-id> with your actual subscription ID

# !az account set --subscription "<subscription-id>"

## Step 3: Deploy Azure Resources

Choose one of two deployment options:

### Option A: Automated Deployment (Recommended)
Uses our Bicep template to deploy all resources at once.

### Option B: Manual Configuration
If you already have Azure resources or prefer manual setup.

---

### Resources to be deployed:

| Resource | Purpose |
|----------|----------|
| Azure OpenAI | GPT-4.1, GPT-4.1-mini, text-embedding-3-large |
| Azure AI Search | Vector + semantic search |
| Azure AI Services | Document Intelligence + Content Understanding |
| Storage Account | Document and figure storage |

**Region**: `swedencentral` (required for Content Understanding GA)

### Option A: Automated Deployment

Run the cells below to deploy all Azure resources using our Bicep template.

In [ ]:
# Configuration - Modify these if needed
RESOURCE_GROUP = "rg-rag-workshop"
LOCATION = "swedencentral"  # Required for Content Understanding GA
BASE_NAME = "ragworkshop"

print("📋 Deployment Configuration")
print("=" * 50)
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Location: {LOCATION}")
print(f"Base Name: {BASE_NAME}")
print("\n⚠️ Review the configuration above before proceeding.")
print("   Modify the variables in this cell if needed.")

In [ ]:
import subprocess
import json
from pathlib import Path

print("🚀 Starting Azure Deployment...")
print("=" * 50)
print("⏱️ This will take approximately 5-10 minutes.\n")

# Get the path to the Bicep file
bicep_path = Path("../../infra/main.bicep").resolve()
print(f"📄 Using Bicep template: {bicep_path}")

if not bicep_path.exists():
    print(f"❌ Bicep file not found at: {bicep_path}")
    raise FileNotFoundError(f"Bicep file not found: {bicep_path}")

# Step 1: Create Resource Group
print(f"\n📦 Creating resource group: {RESOURCE_GROUP}...")
result = subprocess.run(
    ["az", "group", "create", "--name", RESOURCE_GROUP, "--location", LOCATION],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("   ✅ Resource group created/confirmed")
else:
    print(f"   ❌ Failed: {result.stderr}")
    raise Exception(f"Failed to create resource group: {result.stderr}")

# Step 2: Deploy Bicep template
print(f"\n🔧 Deploying Azure resources...")
print("   This may take several minutes...")

result = subprocess.run(
    [
        "az", "deployment", "group", "create",
        "--resource-group", RESOURCE_GROUP,
        "--template-file", str(bicep_path),
        "--parameters", f"baseName={BASE_NAME}", f"location={LOCATION}",
        "--query", "properties.outputs",
        "--output", "json"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("   ✅ Deployment completed successfully!")
    deployment_outputs = json.loads(result.stdout)
    print("\n📋 Deployed Resources:")
    for key in deployment_outputs:
        if 'Endpoint' in key:
            print(f"   • {key}: {deployment_outputs[key]['value']}")
else:
    print(f"   ❌ Deployment failed!")
    print(f"   Error: {result.stderr}")
    raise Exception(f"Deployment failed: {result.stderr}")

In [ ]:
# Generate .env file from deployment outputs
from pathlib import Path
from datetime import datetime
import subprocess
import json

print("📝 Generating .env file...")
print("=" * 50)

# Get subscription ID
sub_result = subprocess.run(
    ["az", "account", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True
)
subscription_id = sub_result.stdout.strip()

# Check if deployment_outputs exists from previous cell
try:
    outputs = deployment_outputs
except NameError:
    # Re-fetch deployment outputs
    print("   Fetching deployment outputs...")
    result = subprocess.run(
        [
            "az", "deployment", "group", "show",
            "--resource-group", RESOURCE_GROUP,
            "--name", "main",
            "--query", "properties.outputs",
            "--output", "json"
        ],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        outputs = json.loads(result.stdout)
    else:
        print(f"❌ Could not fetch deployment outputs: {result.stderr}")
        raise Exception("Run the deployment cell first")

# Extract values
env_content = f'''# ===========================================
# RAG Workshop Environment Configuration
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Region: {LOCATION}
# ===========================================

# Azure Subscription & Resource Group
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={RESOURCE_GROUP}
AZURE_LOCATION={LOCATION}

# Azure OpenAI
AZURE_OPENAI_ENDPOINT={outputs['openAIEndpoint']['value']}
AZURE_OPENAI_API_KEY={outputs['openAIKey']['value']}
AZURE_OPENAI_API_VERSION=2024-08-01-preview
AZURE_OPENAI_DEPLOYMENT_GPT41=gpt-4.1
AZURE_OPENAI_DEPLOYMENT_GPT41_MINI=gpt-4.1-mini
AZURE_OPENAI_DEPLOYMENT_EMBEDDING=text-embedding-3-large

# Azure AI Search
AZURE_SEARCH_ENDPOINT={outputs['searchServiceEndpoint']['value']}
AZURE_SEARCH_API_KEY={outputs['searchServiceAdminKey']['value']}
AZURE_SEARCH_INDEX_NAME=rag-workshop-index

# Azure AI Document Intelligence & Content Understanding
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={outputs['aiServicesEndpoint']['value']}
AZURE_DOCUMENT_INTELLIGENCE_KEY={outputs['aiServicesKey']['value']}
AZURE_CONTENT_UNDERSTANDING_ENDPOINT={outputs['aiServicesEndpoint']['value']}
AZURE_CONTENT_UNDERSTANDING_KEY={outputs['aiServicesKey']['value']}
AZURE_CONTENT_UNDERSTANDING_API_VERSION=2025-11-01

# Azure Storage
AZURE_STORAGE_CONNECTION_STRING={outputs['storageAccountConnectionString']['value']}
AZURE_STORAGE_CONTAINER_DOCUMENTS=documents
AZURE_STORAGE_CONTAINER_FIGURES=figures

# GraphRAG (uses Azure OpenAI settings)
GRAPHRAG_API_KEY=${{AZURE_OPENAI_API_KEY}}
GRAPHRAG_API_BASE=${{AZURE_OPENAI_ENDPOINT}}
GRAPHRAG_API_VERSION=${{AZURE_OPENAI_API_VERSION}}
'''

# Write .env file to project root
env_path = Path("../../.env").resolve()
env_path.write_text(env_content)

print(f"\n✅ .env file created at: {env_path}")
print("\n⚠️ IMPORTANT: The .env file contains sensitive API keys.")
print("   It is already in .gitignore - DO NOT commit it to git!")
print("\n➡️ Proceed to Step 4 or run health-check.ipynb to validate.")

### Option B: Manual Configuration

If you already have Azure resources or prefer manual setup, run the cell below to create your `.env` file interactively.

In [ ]:
# SKIP this cell if you used Option A (automated deployment)

from pathlib import Path
from datetime import datetime

print("📝 Manual Environment Configuration")
print("=" * 50)
print("\nEnter your Azure resource details below.")
print("Leave blank and press Enter to use the default value in brackets.\n")

def get_input(prompt, default=""):
    display = f"{prompt} [{default}]: " if default else f"{prompt}: "
    value = input(display).strip()
    return value if value else default

# Collect values
print("--- Azure Subscription ---")
subscription_id = get_input("Subscription ID")
resource_group = get_input("Resource Group", "rg-rag-workshop")
location = get_input("Location", "swedencentral")

print("\n--- Azure OpenAI ---")
openai_endpoint = get_input("OpenAI Endpoint (e.g., https://xxx.openai.azure.com/)")
openai_key = get_input("OpenAI API Key")

print("\n--- Azure AI Search ---")
search_endpoint = get_input("Search Endpoint (e.g., https://xxx.search.windows.net)")
search_key = get_input("Search Admin Key")

print("\n--- Azure AI Services (Document Intelligence) ---")
di_endpoint = get_input("AI Services Endpoint (e.g., https://xxx.cognitiveservices.azure.com/)")
di_key = get_input("AI Services Key")

print("\n--- Azure Storage ---")
storage_conn = get_input("Storage Connection String")

# Generate .env content
env_content = f'''# ===========================================
# RAG Workshop Environment Configuration
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Region: {location}
# ===========================================

# Azure Subscription & Resource Group
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={resource_group}
AZURE_LOCATION={location}

# Azure OpenAI
AZURE_OPENAI_ENDPOINT={openai_endpoint}
AZURE_OPENAI_API_KEY={openai_key}
AZURE_OPENAI_API_VERSION=2024-08-01-preview
AZURE_OPENAI_DEPLOYMENT_GPT41=gpt-4.1
AZURE_OPENAI_DEPLOYMENT_GPT41_MINI=gpt-4.1-mini
AZURE_OPENAI_DEPLOYMENT_EMBEDDING=text-embedding-3-large

# Azure AI Search
AZURE_SEARCH_ENDPOINT={search_endpoint}
AZURE_SEARCH_API_KEY={search_key}
AZURE_SEARCH_INDEX_NAME=rag-workshop-index

# Azure AI Document Intelligence & Content Understanding
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={di_endpoint}
AZURE_DOCUMENT_INTELLIGENCE_KEY={di_key}
AZURE_CONTENT_UNDERSTANDING_ENDPOINT={di_endpoint}
AZURE_CONTENT_UNDERSTANDING_KEY={di_key}
AZURE_CONTENT_UNDERSTANDING_API_VERSION=2025-11-01

# Azure Storage
AZURE_STORAGE_CONNECTION_STRING={storage_conn}
AZURE_STORAGE_CONTAINER_DOCUMENTS=documents
AZURE_STORAGE_CONTAINER_FIGURES=figures

# GraphRAG (uses Azure OpenAI settings)
GRAPHRAG_API_KEY=${{AZURE_OPENAI_API_KEY}}
GRAPHRAG_API_BASE=${{AZURE_OPENAI_ENDPOINT}}
GRAPHRAG_API_VERSION=${{AZURE_OPENAI_API_VERSION}}
'''

# Write .env file
env_path = Path("../../.env").resolve()
env_path.write_text(env_content)

print(f"\n✅ .env file created at: {env_path}")
print("\n➡️ Proceed to Step 4 to validate your setup.")

## Step 4: Install Python Dependencies

Let's install the required Python packages for the workshop.

In [ ]:
# Install required packages
print("📦 Installing Python dependencies...")
print("=" * 50)
print("⏱️ This may take a few minutes.\n")

!pip install -q -r ../../requirements.txt

print("\n✅ Dependencies installed!")

## Step 5: Quick Validation

Let's do a quick test to ensure your environment is correctly configured.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

print("🔍 Loading and Validating Environment...")
print("=" * 50)

# Load .env file
env_path = Path("../../.env").resolve()
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from: {env_path}")
else:
    print(f"❌ .env file not found at: {env_path}")
    print("   Please run Step 3 first.")
    raise FileNotFoundError(".env file not found")

# Check required environment variables
required_vars = [
    ("AZURE_OPENAI_ENDPOINT", "Azure OpenAI"),
    ("AZURE_OPENAI_API_KEY", "Azure OpenAI"),
    ("AZURE_SEARCH_ENDPOINT", "Azure AI Search"),
    ("AZURE_SEARCH_API_KEY", "Azure AI Search"),
    ("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT", "Document Intelligence"),
    ("AZURE_DOCUMENT_INTELLIGENCE_KEY", "Document Intelligence"),
]

print("\n📋 Checking Environment Variables:")
all_present = True
for var_name, service in required_vars:
    value = os.getenv(var_name)
    if value:
        # Mask sensitive values
        if 'KEY' in var_name or 'SECRET' in var_name:
            display = value[:8] + "..." + value[-4:] if len(value) > 12 else "***"
        else:
            display = value[:50] + "..." if len(value) > 50 else value
        print(f"   ✅ {var_name}: {display}")
    else:
        print(f"   ❌ {var_name}: NOT SET ({service})")
        all_present = False

if all_present:
    print("\n🎉 All required environment variables are set!")
    print("\n➡️ Run health-check.ipynb for detailed connectivity tests.")
else:
    print("\n⚠️ Some environment variables are missing.")
    print("   Please check your .env file or re-run Step 3.")

In [ ]:
# Quick connectivity test - Azure OpenAI
from openai import AzureOpenAI

print("🔗 Testing Azure OpenAI Connection...")
print("=" * 50)

try:
    client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    )
    
    # Test with a simple completion
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1"),
        messages=[
            {"role": "user", "content": "Say 'Hello, RAG Workshop!' in exactly those words."}
        ],
        max_tokens=20
    )
    
    print(f"\n✅ Azure OpenAI (GPT-4.1) is working!")
    print(f"   Response: {response.choices[0].message.content}")
    
except Exception as e:
    print(f"\n❌ Azure OpenAI connection failed!")
    print(f"   Error: {str(e)}")
    print("\n💡 Troubleshooting:")
    print("   1. Verify AZURE_OPENAI_ENDPOINT is correct")
    print("   2. Check that gpt-4.1 deployment exists")
    print("   3. Ensure API key has access to the resource")

## 🎉 Setup Complete!

Congratulations! Your environment is ready for the RAG Workshop.

### Next Steps

1. **Run the full health check**: Open and run `health-check.ipynb` to validate all services
2. **Start Module 1**: Proceed to `../module-1-naive-rag/` to begin the workshop

### Quick Reference

| Resource | Status |
|----------|--------|
| Azure OpenAI | ✅ Configured |
| Azure AI Search | ✅ Configured |
| Document Intelligence | ✅ Configured |
| Content Understanding | ✅ Configured |
| Storage Account | ✅ Configured |

### Troubleshooting

If you encounter issues:
1. Run `health-check.ipynb` for detailed diagnostics
2. Check the Azure Portal for resource status
3. Verify your `.env` file has correct values

---

**[Next Module →](../module-1-naive-rag/README.md)** The Problem with Naive RAG